### Exercise 1: Hill-Climbing Search for Feature Selection in Predictive Modeling

**Problem Statement:** You are given a dataset (select a random dataset) with multiple input features and a target variable. Each state represents a subset of selected features. The goal is to find a feature subset that maximizes model accuracy.

**State Representation**
* A binary vector indicating whether a feature is selected (1) or not (0)

**Initial State**
* A randomly generated feature subset

**Neighbor Generation**
* Flip one bit (add/remove one feature)

**Evaluation Function**
* Cross-validation accuracy of a simple classifier (e.g., Logistic Regression)

**Algorithm Implementation**
* Implement steepest-ascent hill climbing
* Stop when no neighbor improves the current solution

**Experimentation**
* Run the algorithm multiple times with different random initial states

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

# Generate synthetic dataset
X, y = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=4,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    shuffle=True
)

# Create DataFrame
feature_names = [f'X{i+1}' for i in range(10)]
df = pd.DataFrame(X, columns=feature_names)
df['Target'] = y

In [2]:
df

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,Target
0,-0.502512,-1.004990,0.659612,2.026140,-0.141587,-0.424701,-1.394031,-0.125663,1.339076,0.162949,0
1,0.011617,-1.394171,1.124517,1.613316,-0.894583,-0.507344,-1.139049,-0.896129,1.493875,-0.928147,0
2,-0.488427,-1.136354,0.306577,-1.187400,-1.500104,-0.113691,-1.170443,0.436733,0.599881,-0.148213,1
3,0.854666,-0.477392,0.557024,-0.670834,-0.491288,-1.509416,-0.775406,-1.330610,-0.273864,-0.748603,0
4,0.405349,-0.544203,1.555705,1.045939,-0.932586,-0.473783,-0.860063,-1.551221,1.656632,-2.009187,0
...,...,...,...,...,...,...,...,...,...,...,...
495,0.514205,0.442427,0.043351,-0.114982,0.207057,0.556551,0.675046,1.684718,1.501649,-0.425330,1
496,-0.873750,0.386750,-0.082266,-0.791085,0.064489,0.085169,-1.519323,1.556335,0.859150,0.873890,1
497,-1.098178,-0.297053,0.407330,-0.123931,-1.222420,-0.125203,-0.263141,-0.360585,0.396009,-0.603970,0
498,0.611217,0.767042,-0.750241,-0.786072,-0.692019,1.062730,-1.683519,-1.598640,-2.959933,2.007494,0


In [3]:
bin_vec = np.random.randint(0, 2, 10)
bin_vec

array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])

In [4]:
def generate_neighbors(bin_vec, count=10):
    neighbors = []
    for i in range(count):
        neighbors.append(bin_vec.copy())
        neighbors[-1][i] = (neighbors[-1][i] + 1) % 2

    return neighbors

generate_neighbors(bin_vec)

[array([0, 1, 0, 0, 1, 1, 0, 1, 0, 0]),
 array([1, 0, 0, 0, 1, 1, 0, 1, 0, 0]),
 array([1, 1, 1, 0, 1, 1, 0, 1, 0, 0]),
 array([1, 1, 0, 1, 1, 1, 0, 1, 0, 0]),
 array([1, 1, 0, 0, 0, 1, 0, 1, 0, 0]),
 array([1, 1, 0, 0, 1, 0, 0, 1, 0, 0]),
 array([1, 1, 0, 0, 1, 1, 1, 1, 0, 0]),
 array([1, 1, 0, 0, 1, 1, 0, 0, 0, 0]),
 array([1, 1, 0, 0, 1, 1, 0, 1, 1, 0]),
 array([1, 1, 0, 0, 1, 1, 0, 1, 0, 1])]

In [5]:
X = df.drop(["Target"], axis=1).to_numpy()
y = df["Target"].to_numpy()

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

def train_and_eval_lr(X, y, sel_vec):
    sel_array = [i for i, val in enumerate(sel_vec) if val == 1]
    X_sel = X[:, sel_array]

    lr = LogisticRegression(max_iter=1000)
    scores = cross_val_score(lr, X_sel, y, cv=5)
    return round(scores.mean(),2)

In [7]:
ascend = True
iteration = 0

while ascend:
    main_score = train_and_eval_lr(X, y, bin_vec)
    neighbors = generate_neighbors(bin_vec)
    neighbors_scores = []
    next_idx = -1
    max_score = main_score
    for i, neighbor in enumerate(neighbors):
        neighbor_score = train_and_eval_lr(X, y, neighbor)
        if neighbor_score > max_score: 
            max_score = neighbor_score
            next_idx = i
        neighbors_scores.append(neighbor_score)
    if next_idx == -1:
        ascend = False
    else:
        bin_vec = neighbors[next_idx]
    print(f"Iteration: {iteration} | Main Score: {main_score} | Neighbors Scores: {neighbors_scores} | Next Index: {next_idx}")
    iteration += 1

Iteration: 0 | Main Score: 0.88 | Neighbors Scores: [0.88, 0.88, 0.87, 0.88, 0.87, 0.88, 0.91, 0.53, 0.88, 0.9] | Next Index: 6
Iteration: 1 | Main Score: 0.91 | Neighbors Scores: [0.91, 0.91, 0.92, 0.91, 0.91, 0.91, 0.88, 0.72, 0.91, 0.92] | Next Index: 2
Iteration: 2 | Main Score: 0.92 | Neighbors Scores: [0.92, 0.92, 0.91, 0.92, 0.92, 0.92, 0.87, 0.73, 0.91, 0.91] | Next Index: -1


### Exercise 2: Simulated Annealing for Vehicle Routing Optimization
**Problem Statement:** A delivery vehicle must visit a set of locations and return to the depot. The order of visiting locations affects the total travel cost. Each state represents a permutation of delivery locations.

**State Representation**
* A sequence representing the visiting order of delivery points

**Initial State**
* A random permutation of locations

**Neighbor Generation**
* Swap two locations in the route

**Cost Function**
* Total travel distance of the route

**Simulated Annealing Implementation**
* Implement SA using:
    * Temperature schedule T(t)
    * Acceptance probability: P = e<sup>-ΔE/T</sup>

In [75]:
import numpy as np
import pandas as pd

def generate_tsp_data(num_cities=15, map_size=100):
    np.random.seed(42)
    coords = np.random.randint(0, map_size, size=(num_cities, 2))
    
    locations = [f"L{i}" for i in range(num_cities)]
    
    mappings = {city: i for i, city in enumerate(locations)}

    dist_matrix = []
    for i in range(num_cities):
        row = []
        for j in range(num_cities):
            if i == j:
                row.append(0)
            else:
                dist = np.linalg.norm(coords[i] - coords[j])
                row.append(round(dist, 2))
        dist_matrix.append(row)
        
    return locations, mappings, dist_matrix, coords

In [76]:
locations, mappings, distance_matrix, coords = generate_tsp_data(15)

current_route = locations.copy()
import random
random.shuffle(current_route)

In [77]:
def route_cost(route):
    total = 0
    for i in range(len(route) - 1):
        a = mappings[route[i]]
        b = mappings[route[i + 1]]
        total += distance_matrix[a][b]
        
    total += distance_matrix[mappings[route[-1]]][mappings[route[0]]]
    return total

In [78]:
import random

def get_neighbor(route):
    neighbor = route.copy()
    i, j = random.sample(range(len(route)), 2)
    neighbor[i], neighbor[j] = neighbor[j], neighbor[i]
    return neighbor

In [79]:
import math

def simulated_annealing(
    initial_route,
    initial_temp=10000,
    cooling_rate=0.999,
    min_temp=0.001,
    max_iterations=10000
):
    current_route = initial_route
    current_cost = route_cost(current_route)
    
    best_route = current_route
    best_cost = current_cost
    
    T = initial_temp
    iteration = 0

    while T > min_temp and iteration < max_iterations:
        neighbor = get_neighbor(current_route)
        neighbor_cost = route_cost(neighbor)
        
        delta = neighbor_cost - current_cost
        
        if delta < 0 or random.random() < math.exp(-delta / T):
            current_route = neighbor
            current_cost = neighbor_cost
            
            if current_cost < best_cost:
                best_route = current_route
                best_cost = current_cost
        
        T *= cooling_rate
        iteration += 1
    
    return best_route, best_cost

In [80]:
best_route, best_cost = simulated_annealing(current_route)

print("Initial Route:", current_route)
print("Initial Cost:", route_cost(current_route))
print("\nBest Route Found:", best_route)
print("Best Cost:", best_cost)

Initial Route: ['L6', 'L10', 'L5', 'L2', 'L12', 'L8', 'L13', 'L7', 'L0', 'L1', 'L11', 'L9', 'L4', 'L14', 'L3']
Initial Cost: 894.5399999999998

Best Route Found: ['L11', 'L2', 'L14', 'L4', 'L3', 'L5', 'L0', 'L12', 'L1', 'L8', 'L10', 'L7', 'L9', 'L6', 'L13']
Best Cost: 340.23999999999995


### Exercise 3 (Revised): Genetic Algorithm for Autonomous Drone Path Planning

**Problem Statement:**

A drone must travel from a start location to a destination in a 2D environment containing obstacles.  
Each candidate solution represents a sequence of waypoints defining a possible flight path.  
The objective is to evolve paths that minimize the total travel distance while avoiding obstacles.

**Chromosome Representation:**
* A fixed-length sequence of \( (x, y) \) waypoints

**Initial Population:**
* Randomly generated feasible paths between the start and destination

**Fitness Function:**
* Minimize:
    * Total path length
    * Obstacle collisions (assigned a high penalty)

**Genetic Operators:**
* **Selection:** Tournament selection
* **Crossover:** One-point crossover on the waypoint sequence
* **Mutation:** Random perturbation of waypoint coordinates

**Constraints:**
* Waypoints must remain within the map boundaries
* Paths intersecting obstacles are penalized

**Termination Condition:**
* Fixed number of generations or convergence of the population


In [98]:
MAP_WIDTH = 100
MAP_HEIGHT = 100

START = (0, 0)
GOAL = (100, 100)

OBSTACLES = [
    (40, 40, 10),
    (70, 70, 8),
    (60, 20, 6)
]

In [99]:
POPULATION_SIZE = 30
NUM_WAYPOINTS = 6          
GENERATIONS = 100
TOURNAMENT_SIZE = 3
MUTATION_RATE = 0.2
PENALTY = 1000             

In [100]:
def distance(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def collides(point):
    for ox, oy, r in OBSTACLES:
        if distance(point, (ox, oy)) <= r:
            return True
    return False

In [101]:
def fitness(path):
    total_distance = 0
    penalty = 0

    full_path = [START] + path + [GOAL]

    for i in range(len(full_path) - 1):
        total_distance += distance(full_path[i], full_path[i + 1])

    for point in path:
        if collides(point):
            penalty += PENALTY

    return total_distance + penalty

def generate_individual():
    return [
        (
            random.uniform(0, MAP_WIDTH),
            random.uniform(0, MAP_HEIGHT)
        )
        for _ in range(NUM_WAYPOINTS)
    ]

def initialize_population():
    return [generate_individual() for _ in range(POPULATION_SIZE)]

In [102]:
def tournament_selection(population):
    competitors = random.sample(population, TOURNAMENT_SIZE)
    competitors.sort(key=lambda x: fitness(x))
    return competitors[0]

def crossover(parent1, parent2):
    point = random.randint(1, NUM_WAYPOINTS - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

def mutate(individual):
    for i in range(NUM_WAYPOINTS):
        if random.random() < MUTATION_RATE:
            x, y = individual[i]
            x += random.uniform(-5, 5)
            y += random.uniform(-5, 5)

            x = min(max(x, 0), MAP_WIDTH)
            y = min(max(y, 0), MAP_HEIGHT)

            individual[i] = (x, y)

In [103]:
def genetic_algorithm():
    population = initialize_population()
    best_solution = None
    best_fitness = float("inf")

    for generation in range(GENERATIONS):
        new_population = []

        for _ in range(POPULATION_SIZE // 2):
            parent1 = tournament_selection(population)
            parent2 = tournament_selection(population)

            child1, child2 = crossover(parent1, parent2)
            mutate(child1)
            mutate(child2)

            new_population.extend([child1, child2])

        population = new_population

        current_best = min(population, key=lambda x: fitness(x))
        current_fitness = fitness(current_best)

        if current_fitness < best_fitness:
            best_solution = current_best
            best_fitness = current_fitness

        if generation % 10 == 0:
            print(f"Generation {generation} | Best Fitness: {best_fitness:.2f}")

    return best_solution, best_fitness

In [104]:
best_path, best_cost = genetic_algorithm()

print("\nBest Path (Waypoints):")
for wp in best_path:
    print(wp)

print("\nBest Cost:", best_cost)

Generation 0 | Best Fitness: 260.63
Generation 10 | Best Fitness: 153.32
Generation 20 | Best Fitness: 144.91
Generation 30 | Best Fitness: 141.72
Generation 40 | Best Fitness: 141.50
Generation 50 | Best Fitness: 141.50
Generation 60 | Best Fitness: 141.50
Generation 70 | Best Fitness: 141.50
Generation 80 | Best Fitness: 141.45
Generation 90 | Best Fitness: 141.45

Best Path (Waypoints):
(5.649372149011033, 5.466538829103218)
(17.666992436451995, 17.41552041818089)
(26.05498524505751, 25.205069124490635)
(48.226208858732846, 46.80872094796919)
(84.03607752884392, 82.94992224062972)
(91.83868460009936, 91.54096320928846)

Best Cost: 141.44873819346824
